# AERIS — FD001 Data Preparation, EDA, and Feature Engineering

This is **stage 1** of the AERIS workflow. It loads the NASA C-MAPSS FD001 files,
checks data quality, explores engine behavior, creates the RUL target, and compares
the original sensor features with engineered alternatives.

The notebook writes a stable set of CSV files to `data/processed`. Notebook 2 reads
those files and is responsible only for model training and evaluation.

**Project goal**

The project predicts the Remaining Useful Life (RUL) of an aircraft engine: the number of operating cycles left before the end of its simulated life. The prediction will later use sensor readings, operating settings, and cycle history.

**Purpose of this phase:**

Before building a model, we need to understand the dataset, confirm its quality, create the RUL target, and identify useful or unusual features. This reduces errors and helps us choose appropriate preprocessing steps.

## Step 1 — Environment and raw data

The download runs only when one or more source files are missing. This keeps repeated
runs fast and avoids silently replacing files that are already available.

In [ ]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
import os

working_dir = Path.cwd()
configured_data_dir = os.getenv("AERIS_DATA_DIR")

if configured_data_dir:
    default_data_dir = Path(configured_data_dir)
elif Path("/content").is_dir():
    from google.colab import drive

    drive.mount("/content/drive")
    default_data_dir = Path("/content/drive/MyDrive/AERIS/data")
elif working_dir.name == "notebooks":
    default_data_dir = working_dir.parent / "data"
else:
    default_data_dir = working_dir / "data"

BASE_DATA_DIR = default_data_dir.expanduser().resolve()
RAW_DIR = BASE_DATA_DIR / "raw"
PROCESSED_DIR = BASE_DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

NASA_DATASET_URL = (
    "https://phm-datasets.s3.amazonaws.com/NASA/"
    "6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip"
)
EXPECTED_FILES = ("train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt")

missing_files = [name for name in EXPECTED_FILES if not (RAW_DIR / name).is_file()]

if missing_files:
    print("Downloading the official NASA C-MAPSS archive...")
    with urlopen(NASA_DATASET_URL, timeout=120) as response:
        outer_archive = ZipFile(BytesIO(response.read()))

    nested_archive_name = next(
        name for name in outer_archive.namelist() if name.endswith("CMAPSSData.zip")
    )
    with outer_archive.open(nested_archive_name) as nested_file:
        inner_archive = ZipFile(BytesIO(nested_file.read()))

    for filename in EXPECTED_FILES:
        if filename in missing_files:
            (RAW_DIR / filename).write_bytes(inner_archive.read(filename))

missing_files = [name for name in EXPECTED_FILES if not (RAW_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing source files in {RAW_DIR}: {missing_files}")

print(f"FD001 source data is ready in {RAW_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


**Step 2 — Define the dataset structure and load the files**

The raw C-MAPSS text files do not contain column headers, so meaningful names must be assigned before the data can be interpreted the names are assined according to the reedme file of the data.

**What we discover:** Each engine is a time series (each row is a cycle of someengine )of sensor readings ordered by cycle. Training data continues until failure, while test data stops before failure.

In [ ]:
id_columns=['engine_id','cycle']
setting_columns=['setting_1','setting_2','setting_3']
sensor_columns=[f'sensor{i}'for i in range(1,22)]
columns=id_columns+setting_columns+sensor_columns
print('number of columns',len(columns))
print('columns names',columns)

In [ ]:
train_df=pd.read_csv(RAW_DIR / "train_FD001.txt",sep=r"\s+",header=None,names=columns)
test_df=pd.read_csv(RAW_DIR / "test_FD001.txt",sep=r"\s+",header=None,names=columns)
test_rul_df=pd.read_csv(RAW_DIR / "RUL_FD001.txt",sep=r"\s+",header=None,names=['true_rul'])




In [ ]:
print(train_df.shape)
train_df.head()

In [ ]:
print(test_df.shape)
test_df.head()

In [ ]:
print(test_rul_df.shape)
test_rul_df.head()

**Step 3 — Validate data quality and sequence structure**

 Missing values, duplicated engine-cycle pairs, or incorrectly ordered cycles can produce unreliable analysis and may create time-series leakage later.

In [ ]:

def check_dataset(df,dataset_name):

  missing_values=df.isna().sum().sum()
  duplicate_cycles=df.duplicated(subset=['engine_id','cycle']).sum()
  all_columns_numeric = all(pd.api.types.is_numeric_dtype(df[column])for column in df.columns)
  positive_id=(df['engine_id']>0).all()
  positive_cycles=(df['cycle']>0).all()
  #If cycles are incorrectly ordered previous cycle may actually refer to a future measurement causing data leakage.
  df_cycle_order=(df.groupby('engine_id')['cycle'].apply(lambda c:c.is_monotonic_increasing).all())
  consecutive_cycles=(df.groupby('engine_id')['cycle'].diff().dropna().eq(1).all())
  engine_num=df['engine_id'].nunique()


  print(f"--- {dataset_name} ---")
  print("Missing values:", missing_values)
  print("Duplicate engine-cycle rows:", duplicate_cycles)
  print("All columns numeric:", all_columns_numeric)
  print("Engine IDs positive:", positive_id)
  print("Cycles positive:", positive_cycles)
  print("Cycles ordered:",  df_cycle_order)
  print("Cycles consecutive:", consecutive_cycles)
  print("Number of engines:", engine_num)
  print()








these checks confirm whether the training and testing datasets follow the expected FD001 structure before RUL calculation or modeling.

In [ ]:
check_dataset(train_df, 'Training dataset')
check_dataset(test_df, 'Testing dataset')

print('--- RUL dataset ---')
print('Shape:', test_rul_df.shape)
print('Missing values:', test_rul_df.isna().sum().sum())
print("Negative RUL values:", (test_rul_df["true_rul"] < 0).sum())

**Step 4 - Explore engine lifetimes**

Engines do not all fail after the same number of cycles. Understanding lifetime variation shows the range of degradation histories the model will need to learn.

In [ ]:
engine_life= train_df.groupby('engine_id')['cycle'].max()
engine_life.describe()

as the graph shows Engine lifetimes vary, so the dataset contains both shorter- and longer-lived engines rather than one fixed failure time.

In [ ]:
plt.figure(figsize=(8,4))

sns.histplot(engine_life, bins=15, kde=True)

plt.title("Distribution of Engine Lifetimes")
plt.xlabel("Maximum Number of Cycles")
plt.ylabel("Number of Engines")
plt.show()

## Step 5 — Remove constant sensors and inspect behavior

Sensors with one value throughout the training set contain no predictive signal, so
they are removed from both train and test data. The remaining sensors are plotted for
one engine as a quick behavior check.

In [ ]:
constant_sensors = [
    sensor for sensor in sensor_columns
    if train_df[sensor].nunique(dropna=False) == 1
]

print("Constant sensors:", constant_sensors)

clean_train = train_df.drop(columns=constant_sensors).copy()
clean_test = test_df.drop(columns=constant_sensors).copy()

In [ ]:
engine_1 = clean_train.loc[clean_train["engine_id"] == 1]
active_sensors = [sensor for sensor in sensor_columns if sensor not in constant_sensors]

n_columns = 3
n_rows = int(np.ceil(len(active_sensors) / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(15, 3.2 * n_rows), sharex=True)

for axis, sensor in zip(axes.flat, active_sensors):
    axis.plot(engine_1["cycle"], engine_1[sensor], linewidth=1)
    axis.set_title(sensor)
    axis.set_xlabel("Cycle")

for axis in axes.flat[len(active_sensors):]:
    axis.set_visible(False)

fig.suptitle("Sensor behavior for engine 1", y=1.01)
plt.tight_layout()
plt.show()

**Step 6 — Calculate and inspect Remaining Useful Life**
RUL is the numerical target that the machine-learning model will predict.

RUL decreases as an engine advances through its cycles and reaches 0 at its final training record.

In [ ]:
max_cycle= clean_train.groupby('engine_id')['cycle'].transform("max")
clean_train['RUL']=max_cycle-clean_train['cycle']
print( 'number of missing values ',clean_train['RUL'].isna().sum())
clean_train['RUL'].describe()


In [ ]:
plt.figure(figsize=(8, 4))

sns.histplot(
    clean_train["RUL"],
    bins=45
)

plt.title("Distribution of Remaining Useful Life")
plt.xlabel("RUL in Cycles")
plt.ylabel("Number of Records")

plt.show()

**Step 7 — Examine operating settings**

Operating settings should be checked for variation just like sensors. A constant setting cannot help the model explain differences between records.



In [ ]:
print(clean_train[setting_columns].nunique())

setting_3 has only one unique value in FD001, so it is removed consistently from training and testing data.

In [ ]:
clean_train = clean_train.drop(columns=["setting_3"])
clean_test = clean_test.drop(columns=["setting_3"])

## Step 8 — Sensor correlation with RUL

Correlation is used here as an exploratory ranking, not as the final feature-selection
rule. Nonlinear effects are evaluated later with grouped cross-validation.

In [ ]:
available_sensors = [sensor for sensor in sensor_columns if sensor in clean_train.columns]
correlation = clean_train[available_sensors + ["RUL"]].corr()["RUL"].drop("RUL")
correlation = correlation.sort_values(key=np.abs, ascending=False)
correlation

In [ ]:
top_correlations = (
    correlation
    .head(10)
    .sort_values()
)

top_correlations.plot(
    kind="barh",
    figsize=(6,3)
)

plt.title("Top Sensors Correlated with RUL")
plt.xlabel("Absolute Correlation")
plt.ylabel("Sensor")

plt.show()

**Step 9 — Identify potential outliers with the IQR rule**
Why we do it: Unusually high or low readings should be identified before preprocessing. In predictive maintenance, however, an unusual reading may represent real degradation rather than an error.

In [ ]:
Q1=clean_train[available_sensors].quantile(0.25)
Q3=clean_train[available_sensors].quantile(0.75)
IQR=Q3-Q1
lower_bound=Q1-1.5*IQR
upper_bound=Q3+1.5*IQR

outlier_mask=((clean_train[available_sensors]<lower_bound)| (clean_train[available_sensors]>upper_bound))

outlier_count=outlier_mask.sum().sort_values(ascending=False)



In [ ]:
outlier_percentage = (outlier_count / len(clean_train) * 100).sort_values(ascending=False)
outlier_summary = pd.DataFrame({
    "count": outlier_count,
    "percentage": outlier_percentage,
}).sort_values("percentage", ascending=False)

display(outlier_summary)

outlier_summary["percentage"].plot(kind="bar", figsize=(14, 5))
plt.title("IQR outlier percentage by numeric column")
plt.ylabel("Outliers (%)")
plt.tight_layout()
plt.show()

## Step 10 — Feature engineering: changes over time

The IQR check above is diagnostic only. Sensor extremes are retained because they may
represent real degradation rather than bad records.

The three sensors below form an explicit **candidate experiment** based on their visible
degradation behavior and RUL relationship. They are not assumed to be the final answer;
Step 11 compares their engineered features against the complete original feature set.

In [ ]:
selected_sensors = ["sensor2", "sensor3", "sensor7"]

train_features = clean_train.copy()
test_features = clean_test.copy()

### Step 10.1 — Create nine time-based features

**Why:** A single reading describes the current sensor value, but it does not directly describe its recent direction or variation. For each of three sensors, we create:
- **diff:** Measures the change between two consecutive cycles.
- **mean5:** Calculates the average of the last five readings to show the trend.
- **std5:** Measures how much the readings vary over the last five cycles.

This adds 9 features to the original 18, giving us 27 features to test.

In [ ]:

def build_features(df):
    df = df.sort_values(
        ["engine_id", "cycle"]
    ).copy()

    for sensor in selected_sensors:
        group = df.groupby("engine_id")[sensor]

        # Difference from previous cycle
        df[f"{sensor}_diff"] = (
            group.diff().fillna(0)
        )

        # Average of the last 5 readings
        df[f"{sensor}_mean5"] = (
            group.transform(
                lambda x: x.rolling(
                    window=5,
                    min_periods=1
                ).mean()
            )
        )

        # Standard deviation of last 5 readings
        df[f"{sensor}_std5"] = (
            group.transform(
                lambda x: x.rolling(
                    window=5,
                    min_periods=1
                ).std()
            ).fillna(0)
        )

    return df

train_features = build_features(clean_train)
test_features = build_features(clean_test)

print("New training shape:", train_features.shape)
print("New testing shape:", test_features.shape)


### Step 10.2 — Visualize what the new features mean

**Why:** Numbers alone make it hard to see what `diff`, `mean5`, and `std5` represent. We plot one example engine (`engine_id = 1`) and one selected sensor (`sensor2`) to illustrate the transformations.

**Reading the plots:** The top plot compares raw sensor readings with the five-cycle rolling mean. The bottom plot shows cycle-to-cycle changes and local standard deviation. These curves are descriptive, not proof of noise removal or fault detection. The actual prediction benefit is tested in Step 11.

**Expected result:** Two compact plots, rather than many repeated figures.

In [ ]:
engine_example = train_features.loc[train_features["engine_id"] == 1]
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(engine_example["cycle"], engine_example["sensor2"], alpha=0.6, label="Raw sensor2")
axes[0].plot(engine_example["cycle"], engine_example["sensor2_mean5"], label="Rolling mean (5)")
axes[0].set_ylabel("Sensor reading")
axes[0].legend()
axes[1].plot(engine_example["cycle"], engine_example["sensor2_diff"], label="Cycle difference")
axes[1].plot(engine_example["cycle"], engine_example["sensor2_std5"], label="Rolling std (5)")
axes[1].set_xlabel("Cycle")
axes[1].set_ylabel("Derived values")
axes[1].legend()
plt.tight_layout()
plt.show()


### Step 10.3 — Validate the engineered dataset

**Why:** Before evaluating a model, confirm that feature generation did not lose observations, introduce missing values, or accidentally drop the original model inputs.

**Checks:** `original_features` excludes the engine identifier and RUL target, leaving the original 18 inputs (including `cycle`); `engineered_columns` lists the nine new features. `assert` stops execution if a requirement fails. We check train/test row counts, missing values, and that all model input columns exist in both datasets.

**Expected result:** A short confirmation: `Original: 18 features | Engineered: 27 features`. This checks structure, not predictive quality.

In [ ]:
engineered_columns = [
    f"{sensor}_{suffix}"
    for sensor in selected_sensors
    for suffix in ("diff", "mean5", "std5")
]
original_features = [column for column in clean_train.columns if column not in {"engine_id", "RUL"}]

expected_engineered_count = len(selected_sensors) * 3
assert len(engineered_columns) == expected_engineered_count
assert set(engineered_columns).issubset(train_features.columns)
assert set(engineered_columns).issubset(test_features.columns)
assert train_features[original_features + engineered_columns + ["RUL"]].notna().all().all()
assert test_features[original_features + engineered_columns].notna().all().all()

print(f"Original features: {len(original_features)}")
print(f"Engineered features: {len(engineered_columns)}")

# Step 11 — Compare three feature approaches on unseen engines

**Research question:** Do the nine engineered features or PCA improve RUL prediction enough to justify additional complexity?

We evaluate the **same held-out engines**, the **same RUL target**, and the **same Random Forest settings** for all three approaches:

1. **Original (18):** the cleaned sensor/setting/cycle inputs.
2. **Engineered (27):** the original 18 plus the nine time-based features.
3. **PCA (12 components):** scale the original 18 inputs, then transform them into **12 principal components**. PCA is a dimensionality-reduction method, not an additional nine sensor features.

**Why engine-level splitting:** Each engine has many cycles. Splitting individual rows could place cycles from the same engine in both training and validation. `GroupShuffleSplit` keeps each engine wholly in one side.

**Metrics:** Lower **MAE** and **RMSE** are better; higher **R²** is better. MAE is the average absolute RUL prediction error in cycles.

**Scope:** This is a controlled feature comparison, **not** final model training, NASA test evaluation, or hyperparameter tuning.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(clean_train, groups=clean_train["engine_id"]))
assert set(clean_train.iloc[train_idx]["engine_id"]).isdisjoint(set(clean_train.iloc[val_idx]["engine_id"]))
y_train = clean_train["RUL"].iloc[train_idx]
y_val = clean_train["RUL"].iloc[val_idx]
print("Training engines:", clean_train.iloc[train_idx]["engine_id"].nunique())
print("Validation engines:", clean_train.iloc[val_idx]["engine_id"].nunique())


In [ ]:
def evaluate_features(name, x_train, x_val):
    model = RandomForestRegressor(n_estimators=100, max_depth=15,
                                  min_samples_leaf=3, random_state=42, n_jobs=-1)
    model.fit(x_train, y_train)
    pred = model.predict(x_val)
    return {"Approach": name,
            "Features": x_train.shape[1],
            "MAE": mean_absolute_error(y_val, pred),
            "RMSE": np.sqrt(mean_squared_error(y_val, pred)),
            "R2": r2_score(y_val, pred)}

results = []
results.append(evaluate_features("Original", clean_train.iloc[train_idx][original_features],
                                 clean_train.iloc[val_idx][original_features]))
all_features = original_features + engineered_columns
results.append(evaluate_features("Engineered", train_features.iloc[train_idx][all_features],
                                 train_features.iloc[val_idx][all_features]))

scaler = StandardScaler()
x_pca_train_scaled = scaler.fit_transform(clean_train.iloc[train_idx][original_features])
x_pca_val_scaled = scaler.transform(clean_train.iloc[val_idx][original_features])
pca = PCA(n_components=12)
x_pca_train = pca.fit_transform(x_pca_train_scaled)
x_pca_val = pca.transform(x_pca_val_scaled)
results.append(evaluate_features("PCA", x_pca_train, x_pca_val))
comparison = pd.DataFrame(results)
display(comparison.round(3))


### Step 11.1 — Visualize the single-split comparison

**Why:** The table in Step 11 shows exact MAE/RMSE/R² values; a single bar chart makes the MAE differences easier to see.

**How to read:** Shorter bars indicate lower validation MAE. A small difference between bars does not prove that one approach will consistently win on other engines. We therefore also run grouped cross-validation below.

In [ ]:
ax = comparison.plot.bar(x="Approach", y="MAE", legend=False, figsize=(8, 4), rot=0)
ax.set_ylabel("Validation MAE (cycles)")
ax.set_title("Feature approaches — same engine-level validation split")
plt.tight_layout()
plt.show()


### Step 11.2 — Group Cross-Validation (3 folds)

**Why:** A single hold-out result depends on which engines happen to be selected. `GroupKFold(n_splits=3)` evaluates each representation on **three different sets of unseen engines**, keeping all cycles of any engine within one fold.

**Fair comparison:** The same folds, RUL target, **100 trees**, `max_depth=15`, and `min_samples_leaf=3` are used for all three approaches. In our earlier tree-count tuning, 100 trees performed slightly better than 50; we keep 100 fixed here rather than tuning it again. These are **comparison settings**, not the separate final model's hyperparameters.

**Preventing leakage:** For PCA, `StandardScaler` and `PCA(n_components=12)` are placed inside a `Pipeline`. Each fold fits both transformations **only on that fold's training engines**, then applies them to its validation engines.

**What the code reports:** For each approach, the mean MAE, variation in MAE across folds (`MAE Std`), mean RMSE, and mean R². Smaller mean MAE is preferable, but the size and consistency of the differences matter. This cross-validation does **not** use NASA's held-out test engines.

In [ ]:
cv = GroupKFold(n_splits=3)
groups = clean_train["engine_id"]
y = clean_train["RUL"]
feature_sets = {
    "Original (18)": clean_train[original_features],
    "Engineered (27)": train_features[original_features + engineered_columns],
    "PCA (12)": clean_train[original_features],
}

cv_rows = []
for approach, X in feature_sets.items():
    fold_mae, fold_rmse, fold_r2 = [], [], []
    for fold_train, fold_val in cv.split(X, y, groups=groups):
        rf = RandomForestRegressor(
            n_estimators=100, max_depth=15, min_samples_leaf=3,
            random_state=42, n_jobs=-1
        )
        model = (Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=12)),
            ("rf", rf)
        ]) if approach == "PCA (12)" else rf)

        model.fit(X.iloc[fold_train], y.iloc[fold_train])
        pred = model.predict(X.iloc[fold_val])
        fold_mae.append(mean_absolute_error(y.iloc[fold_val], pred))
        fold_rmse.append(np.sqrt(mean_squared_error(y.iloc[fold_val], pred)))
        fold_r2.append(r2_score(y.iloc[fold_val], pred))

    cv_rows.append({
        "Approach": approach,
        "Mean MAE": np.mean(fold_mae),
        "MAE Std": np.std(fold_mae),
        "Mean RMSE": np.mean(fold_rmse),
        "Mean R2": np.mean(fold_r2)
    })

cv_comparison = pd.DataFrame(cv_rows).sort_values("Mean MAE")
display(cv_comparison.round(3))


### Interpretation — Feature Selection

We compared three approaches: Original (18), Engineered (27), and PCA (12).

The 9 engineered features gave only a small improvement. PCA performed slightly better in single validation, but the original features performed better in Cross-Validation.

We chose the **18 original features** because they gave good results, were easier to understand, and kept the original sensor readings without extra transformations.

## Step 12 — Export the processed data contract

Notebook 2 should load these files instead of repeating cleaning or feature engineering.
The metadata file records the target, identifiers, and feature lists so later stages do
not depend on hard-coded feature counts.

In [ ]:
import json

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

exports = {
    "train_original.csv": clean_train,
    "test_original.csv": clean_test,
    "train_engineered.csv": train_features,
    "test_engineered.csv": test_features,
    "test_rul.csv": test_rul_df,
}

for filename, dataframe in exports.items():
    dataframe.to_csv(PROCESSED_DIR / filename, index=False)

metadata = {
    "dataset": "NASA C-MAPSS FD001",
    "target_column": "RUL",
    "id_column": "engine_id",
    "time_column": "cycle",
    "feature_columns": original_features,
    "engineered_features": engineered_columns,
    "selected_sensors": selected_sensors,
    "files": list(exports),
}

(PROCESSED_DIR / "dataset_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print(f"Saved {len(exports)} datasets and metadata to {PROCESSED_DIR}")
for filename in [*exports, "dataset_metadata.json"]:
    print(" -", filename)